In [2]:
import pandas as pd

In [3]:
data = pd.read_csv('../../data/data/processed/fdemand.csv')

In [4]:
# Ensure 'data' DataFrame is loaded and process datetime columns
# Combine date and time components into a single datetime column
data['start_datetime'] = pd.to_datetime(data[['st_year', 'st_month', 'st_day', 'st_hour', 'st_minute']].rename(columns={
    'st_year': 'year', 'st_month': 'month', 'st_day': 'day', 'st_hour': 'hour', 'st_minute': 'minute'
}))
data['start_date'] = data['start_datetime'].dt.date

In [5]:
def get_time_period(hour):
    if 7 <= hour < 19:  # 7 AM to 6:59 PM
        return '7am-7pm'
    else:  # 7 PM to 6:59 AM
        return '7pm-7am'

data['time_period'] = data['st_hour'].apply(get_time_period)

# Calculate departures for 12-hour periods
departures_12hr = data.groupby(['start_station', 'start_date', 'time_period']).size().reset_index(name='departures')
departures_12hr.rename(columns={'start_station': 'station_id'}, inplace=True)

# Calculate arrivals for 12-hour periods
arrivals_12hr = data.groupby(['end_station', 'start_date', 'time_period']).size().reset_index(name='arrivals')
arrivals_12hr.rename(columns={'end_station': 'station_id'}, inplace=True)

# Merge departures and arrivals for 12-hour periods
net_change_12hr_df = pd.merge(
    departures_12hr,
    arrivals_12hr,
    on=['station_id', 'start_date', 'time_period'],
    how='outer'
).fillna(0)

# Calculate net change for 12-hour periods
net_change_12hr_df['net_change'] = net_change_12hr_df['arrivals'] - net_change_12hr_df['departures']

In [6]:
print("Net Change per Station per 12-hour period (first 5 rows):")
display(net_change_12hr_df.tail(20))
display(net_change_12hr_df.shape)

Net Change per Station per 12-hour period (first 5 rows):


,station_id,start_date,time_period,departures,arrivals,net_change
844423,3453,2026-03-29,7am-7pm,1.0,0.0,-1.0
844424,3455,2026-03-14,7am-7pm,2.0,0.0,-2.0
844425,3455,2026-03-15,7am-7pm,0.0,1.0,1.0
844426,3455,2026-03-18,7am-7pm,1.0,0.0,-1.0
844427,3455,2026-03-19,7am-7pm,0.0,1.0,1.0
844428,3455,2026-03-21,7am-7pm,1.0,1.0,0.0
844429,3455,2026-03-23,7am-7pm,0.0,1.0,1.0
844430,3455,2026-03-24,7am-7pm,1.0,1.0,0.0
844431,3455,2026-03-24,7pm-7am,0.0,1.0,1.0
844432,3455,2026-03-25,7am-7pm,0.0,1.0,1.0


(844443, 6)

In [7]:
net_change_12hr_df.to_csv('net_change_12hr.csv', index=False)